# Diagnostics: balance sheet violations and statement layout

Two open questions from `Build_analytics.ipynb`, investigated here rather than
in the build so the deliverable stays a build and this stays an investigation.

## 1. Why do 3.48% of filings fail the balance sheet identity?

`sum_eiendeler` must equal `sum_egenkapital_gjeld` — total assets equal total
equity plus liabilities. That is an accounting constraint, not an assumption
about this dataset, yet 15,462 of 444,645 filings violate it above a 0.5 NOK
threshold.

Four candidate explanations, and what would distinguish them:

| explanation | signature |
|---|---|
| Our flattening is wrong | MongoDB agrees with itself but disagrees with our table |
| Units, thousands vs whole NOK | discrepancies cluster on round multiples |
| One side is internally inconsistent | components of one side fail to sum to its own total |
| Genuine source error | scattered magnitudes, present identically in MongoDB |

The provenance check comes first, because if the fault is ours nothing else
matters. It re-tests the identity directly in MongoDB, server-side, over every
filing — bypassing Parquet, the schema and the flattening entirely.

## 2. What does `oppstillingsplan` tell us, given it never varies?

Every filing carries `oppstillingsplan = "store"`. The layout hypothesis for
missing revenue is therefore untestable as originally framed: a field with no
variance cannot explain variance in anything else.

That is a result rather than a dead end, but it leaves the question open, so the
second half tests the same underlying idea using industry code as a proxy for
the kind of income statement a filer keeps, and looks for the positive signature
of a holding or dormant company rather than only the absence of revenue.

Nothing here writes to the analytics file. Run it after `Build_analytics.ipynb`.

In [1]:
import json
import os

from pyspark.sql import functions as F

from bootstrap import ANALYTICS_FILE, DATA_DIR, PARQUET_DIR, mongo_db, start_spark

spark = start_spark("group13_diagnostics", describe=False)
db = mongo_db(spark)

out = spark.read.parquet(ANALYTICS_FILE)
filed = out.filter("har_regnskap").cache()

# Same rule as the build's verification cell, kept in one place here.
DELTA = F.col("sum_eiendeler") - F.col("sum_egenkapital_gjeld")
TOL = 0.5

bal = filed.filter(F.col("sum_eiendeler").isNotNull()
                   & F.col("sum_egenkapital_gjeld").isNotNull())
viol = bal.filter(F.abs(DELTA) > TOL).cache()

n_bal, n_viol = bal.count(), viol.count()
print("filings with both totals present : %d" % n_bal)
print("violating the identity           : %d  (%.3f%%)" % (n_viol, 100.0 * n_viol / n_bal))

findings = {"tolerance_nok": TOL, "checked": n_bal, "violations": n_viol,
            "violation_pct": 100.0 * n_viol / n_bal}

filings with both totals present : 444646
violating the identity           : 15463  (3.478%)


## Provenance: is this ours or the source's?

The identity is re-tested inside MongoDB, server-side, over every filing. This
path shares nothing with the analytics table — not Parquet, not
`schemas.py`,not the flattening — so agreement means the discrepancy arrived
with the data and disagreement means we introduced it. Counted over all
filings, not a sample.

In [2]:
pipeline = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$project": {
        "a": "$data.eiendeler.sumEiendeler",
        "b": "$data.egenkapitalGjeld.sumEgenkapitalGjeld",
    }},
    {"$match": {"a": {"$ne": None}, "b": {"$ne": None}}},
    {"$group": {
        "_id": None,
        "n": {"$sum": 1},
        "viol": {"$sum": {"$cond": [
            {"$gt": [{"$abs": {"$subtract": ["$a", "$b"]}}, TOL]}, 1, 0]}},
    }},
]
mongo_res = db.financial_data.aggregate(pipeline, allowDiskUse=True).next()

print("MongoDB, both totals present : %d" % mongo_res["n"])
print("MongoDB, violating           : %d  (%.3f%%)"
      % (mongo_res["viol"], 100.0 * mongo_res["viol"] / mongo_res["n"]))
print()
agrees = (mongo_res["n"] == n_bal) and (mongo_res["viol"] == n_viol)
print("Agrees with the analytics table: %s" % ("YES" if agrees else "NO"))
if agrees:
    print("  The discrepancy is present in the source documents. Our Parquet")
    print("  export and flattening did not introduce it.")
else:
    print("  The two disagree. The fault is somewhere in export or flattening,")
    print("  and everything below is measuring our own bug rather than the data.")

findings["mongodb"] = {"checked": mongo_res["n"], "violations": mongo_res["viol"],
                       "agrees_with_analytics": agrees}

MongoDB, both totals present : 445225
MongoDB, violating           : 15478  (3.476%)

Agrees with the analytics table: NO
  The two disagree. The fault is somewhere in export or flattening,
  and everything below is measuring our own bug rather than the data.


## Magnitude

Absolute discrepancy in NOK, and relative to the size of the balance sheet. The
relative figure is what decides whether this matters. A 100 NOK gap on a
billion-NOK balance sheet is noise; the same gap on a 500 NOK balance sheet is
the whole statement.

In [3]:
mag = viol.select(
    F.abs(DELTA).alias("abs_delta"),
    DELTA.alias("signed_delta"),
    F.col("sum_eiendeler").alias("assets"),
    (F.abs(DELTA) / F.greatest(F.abs(F.col("sum_eiendeler")), F.lit(1.0)) * 100).alias("rel_pct"),
).cache()

probs = [0.01, 0.25, 0.5, 0.75, 0.9, 0.99]
labels = ["p1", "p25", "median", "p75", "p90", "p99"]

abs_q = mag.approxQuantile("abs_delta", probs, 0.0)
rel_q = mag.approxQuantile("rel_pct", probs, 0.0)

print("%-8s %18s %14s" % ("", "abs delta (NOK)", "rel to assets"))
print("-" * 44)
for lab, a, r in zip(labels, abs_q, rel_q):
    print("%-8s %18.2f %13.4f%%" % (lab, a, r))

ext = mag.agg(F.min("abs_delta"), F.max("abs_delta"),
              F.min("rel_pct"), F.max("rel_pct")).collect()[0]
print("%-8s %18.2f %13.4f%%" % ("min", ext[0], ext[2]))
print("%-8s %18.2f %13.4f%%" % ("max", ext[1], ext[3]))

# Direction: a consistent sign would point at a systematic omission on one side.
n_pos = mag.filter("signed_delta > 0").count()
print()
print("assets exceed equity+liabilities : %d  (%.1f%%)" % (n_pos, 100.0 * n_pos / n_viol))
print("equity+liabilities exceed assets : %d  (%.1f%%)"
      % (n_viol - n_pos, 100.0 * (n_viol - n_pos) / n_viol))

findings["magnitude"] = {
    "abs_delta_quantiles": dict(zip(labels, abs_q)),
    "rel_pct_quantiles": dict(zip(labels, rel_q)),
    "abs_delta_min": float(ext[0]), "abs_delta_max": float(ext[1]),
    "assets_exceed_liabilities": n_pos,
    "liabilities_exceed_assets": n_viol - n_pos,
}

            abs delta (NOK)  rel to assets
--------------------------------------------
p1                     1.00        0.0000%
p25                    1.00        0.0000%
median                 1.00        0.0001%
p75                    1.00        0.0007%
p90                 2000.00        0.6104%
p99              1159611.00  3000000.0000%
min                    1.00        0.0000%
max          73378000000.00 810000000000.0000%

assets exceed equity+liabilities : 7577  (49.0%)
equity+liabilities exceed assets : 7886  (51.0%)


## Units test

If the register mixed whole NOK with thousands, the discrepancies would land on
round multiples far more often than chance allows. Scattered residues point at
something else.

The reference point matters: about 0.1% of arbitrary values are multiples of
1000 by chance, so anything near that rate is not evidence of a units problem.

In [4]:
rounded = mag.select(
    (F.col("abs_delta") == F.round("abs_delta")).alias("whole"),
    (F.col("abs_delta") % 100 == 0).alias("mult_100"),
    (F.col("abs_delta") % 1000 == 0).alias("mult_1000"),
    (F.col("abs_delta") % 1000000 == 0).alias("mult_1e6"),
).agg(*[F.sum(F.col(c).cast("long")).alias(c)
        for c in ["whole", "mult_100", "mult_1000", "mult_1e6"]]).collect()[0]

print("%-14s %10s %9s   %s" % ("multiple of", "count", "share", "expected by chance"))
print("-" * 62)
for col, chance in [("whole", "n/a"), ("mult_100", "1.0%"),
                    ("mult_1000", "0.1%"), ("mult_1e6", "0.0001%")]:
    print("%-14s %10d %8.2f%%   %s"
          % (col, rounded[col], 100.0 * rounded[col] / n_viol, chance))

findings["rounding"] = {c: int(rounded[c]) for c in
                        ["whole", "mult_100", "mult_1000", "mult_1e6"]}

multiple of         count     share   expected by chance
--------------------------------------------------------------
whole               15463   100.00%   n/a
mult_100             1046     6.76%   1.0%
mult_1000             981     6.34%   0.1%
mult_1e6               44     0.28%   0.0001%


## Which side is internally in consistent?

Each side of the balance sheet has its own components, so each can be checked
against its own total independently of the other:- assets: `omloepsmidler +
anleggsmidler ` should equal `sum_eiendeler`- equity and liabilities:
`sum_egenkapital + sum_gjeld ` should equal `sum_egenkapital_gjeld` If one side
sums correctly and the other does not, the fault is localised. If both sum
correctly, then each side is internally coherent and they simply disagree with
each other, which points at the source rather than at a droppedcomponent.

Only rows where every component is present are counted, so a null is never
silently read as a zero.

In [5]:
assets_ok = (F.abs(F.col("omloepsmidler") + F.col("anleggsmidler")
                   - F.col("sum_eiendeler")) <= TOL)
liab_ok = (F.abs(F.col("sum_egenkapital") + F.col("sum_gjeld")
                 - F.col("sum_egenkapital_gjeld")) <= TOL)

a_full = viol.filter(F.col("omloepsmidler").isNotNull()
                     & F.col("anleggsmidler").isNotNull())
l_full = viol.filter(F.col("sum_egenkapital").isNotNull()
                     & F.col("sum_gjeld").isNotNull())

n_a, n_a_ok = a_full.count(), a_full.filter(assets_ok).count()
n_l, n_l_ok = l_full.count(), l_full.filter(liab_ok).count()

print("Among the %d violating filings:" % n_viol)
print("  assets side components present   : %d" % n_a)
print("    and summing to their own total : %d  (%.1f%%)"
      % (n_a_ok, 100.0 * n_a_ok / n_a if n_a else 0))
print("  equity/liab components present   : %d" % n_l)
print("    and summing to their own total : %d  (%.1f%%)"
      % (n_l_ok, 100.0 * n_l_ok / n_l if n_l else 0))

# Same check on conforming filings, as a control. Without it there is no way to
# tell whether a low rate is characteristic of the violators or of the corpus.
ok_rows = bal.filter(F.abs(DELTA) <= TOL)
c_a = ok_rows.filter(F.col("omloepsmidler").isNotNull() & F.col("anleggsmidler").isNotNull())
c_l = ok_rows.filter(F.col("sum_egenkapital").isNotNull() & F.col("sum_gjeld").isNotNull())
c_a_n, c_a_ok = c_a.count(), c_a.filter(assets_ok).count()
c_l_n, c_l_ok = c_l.count(), c_l.filter(liab_ok).count()

print()
print("Control, the %d conforming filings:" % ok_rows.count())
print("  assets side sums correctly       : %d of %d  (%.1f%%)"
      % (c_a_ok, c_a_n, 100.0 * c_a_ok / c_a_n if c_a_n else 0))
print("  equity/liab side sums correctly  : %d of %d  (%.1f%%)"
      % (c_l_ok, c_l_n, 100.0 * c_l_ok / c_l_n if c_l_n else 0))

findings["side_consistency"] = {
    "violating": {"assets_checked": n_a, "assets_ok": n_a_ok,
                  "liab_checked": n_l, "liab_ok": n_l_ok},
    "conforming": {"assets_checked": c_a_n, "assets_ok": c_a_ok,
                   "liab_checked": c_l_n, "liab_ok": c_l_ok},
}

Among the 15463 violating filings:
  assets side components present   : 13192
    and summing to their own total : 12318  (93.4%)
  equity/liab components present   : 14682
    and summing to their own total : 13408  (91.3%)

Control, the 429183 conforming filings:
  assets side sums correctly       : 333969 of 374763  (89.1%)
  equity/liab side sums correctly  : 344064 of 413900  (83.1%)


## Does it cluster?

Rates are reported against each group's own base, not as a share of all
violations, so a large legal form does not look like a culprit merely for being
large.

In [6]:
def rate_by(column, limit=12, tol=TOL):
    """Violation rate within each value of `column`, largest groups first."""
    marked = bal.withColumn("_viol", (F.abs(DELTA) > tol).cast("int"))
    rows = (marked.groupBy(column)
            .agg(F.count(F.lit(1)).alias("n"), F.sum("_viol").alias("viol"))
            .orderBy(F.desc("n")).limit(limit).collect())
    print("%-22s %10s %10s %9s" % (column, "filings", "violations", "rate"))
    print("-" * 54)
    result = {}
    for r in rows:
        key = str(r[column])
        pct = 100.0 * r["viol"] / r["n"]
        result[key] = {"n": r["n"], "violations": r["viol"], "pct": pct}
        print("%-22s %10d %10d %8.2f%%" % (key, r["n"], r["viol"], pct))
    print()
    return result


findings["clustering"] = {}
for col in ["organisasjonsform_kode", "regnskapsaar", "valuta",
            "smaa_foretak", "avviklingsregnskap", "is_full_year",
            "morselskap", "ikke_revidert"]:
    findings["clustering"][col] = rate_by(col)

organisasjonsform_kode    filings violations      rate
------------------------------------------------------
AS                         403782      13496     3.34%
ESEK                        10324        354     3.43%
BRL                          9995        171     1.71%
STI                          5680        396     6.97%
NUF                          3545        217     6.12%
ENK                          3266        188     5.76%
FLI                          2810        260     9.25%
SA                           1900        100     5.26%
DA                           1211         74     6.11%
ANS                           897         44     4.91%
SAM                           404         12     2.97%
ASA                           195         63    32.31%

regnskapsaar              filings violations      rate
------------------------------------------------------
2025                       417897      12713     3.04%
2024                        19133       2158    11.28%
2023     

## Samples

Ten violating filings, spanning the magnitude range, so a few can be checked by
hand against the Regnskapsregisteret API. Nothing here proves anything on its
own; it is material for a manual spot check.

In [7]:
sample = (viol.select("organisasjonsnummer", "navn", "organisasjonsform_kode",
                      "regnskapsaar", "sum_eiendeler", "sum_egenkapital_gjeld",
                      "sum_egenkapital", "sum_gjeld",
                      DELTA.alias("delta"))
          .orderBy(F.desc(F.abs(DELTA))).limit(5))
print("Largest discrepancies:")
sample.show(5, truncate=30)

print("Smallest discrepancies above tolerance:")
(viol.select("organisasjonsnummer", "navn", "sum_eiendeler",
             "sum_egenkapital_gjeld", DELTA.alias("delta"))
     .orderBy(F.abs(DELTA)).limit(5).show(5, truncate=30))

findings["samples"] = [r.asDict() for r in sample.collect()]
print("API check: https://data.brreg.no/regnskapsregisteret/regnskap/<organisasjonsnummer>")

Largest discrepancies:
+-------------------+-----------------------------+----------------------+------------+-------------+---------------------+---------------+----------+-----------+
|organisasjonsnummer|                         navn|organisasjonsform_kode|regnskapsaar|sum_eiendeler|sum_egenkapital_gjeld|sum_egenkapital| sum_gjeld|      delta|
+-------------------+-----------------------------+----------------------+------------+-------------+---------------------+---------------+----------+-----------+
|          982463718|                  TELENOR ASA|                   ASA|        2024|   2.11512E11|           1.38134E11|     1.09129E11| 2.9005E10|  7.3378E10|
|          986228608|       YARA INTERNATIONAL ASA|                   ASA|        2024|    7.9575E10|           1.00938E11|      2.7813E10| 7.3125E10| -2.1363E10|
|          984115113|                 LIONHEART AS|                    AS|        2024|    1.95816E8|           9.662478E9|     4.657592E9|5.004886E9|-9.466662E9|

# Re-analysis at a corrected tolerance

The magnitude output above changes what the earlier numbers mean, so everything
from here re-derives them.

Every delta is a whole number and the median is 1 NOK. The Regnskapsregisteret
reports whole kroner, so two independently rounded subtotals can differ by one
as a matter of arithmetic rather than error. A 0.5 NOK threshold therefore
counts rounding as a violation, which was a mistake in the original check: the
threshold was chosen without first measuring the precision of the source. This
section finds a defensible threshold, then decomposes what survives it.

In [8]:
# Absolute thresholds. 1.5 is the first that cannot be reached by rounding two
# whole-NOK subtotals independently.
print("%-14s %10s %9s" % ("abs tolerance", "violations", "of filings"))
print("-" * 36)
tol_scan = {}
for t in [0.5, 1.5, 2.5, 10.0, 100.0, 1000.0, 100000.0]:
    c = bal.filter(F.abs(DELTA) > t).count()
    tol_scan[str(t)] = c
    print("%14.1f %10d %8.3f%%" % (t, c, 100.0 * c / n_bal))

# A relative threshold scales with the size of the balance sheet, so a 1 NOK gap
# on a billion-NOK filing and on a 500 NOK filing are treated differently.
REL = F.abs(DELTA) / F.greatest(F.abs(F.col("sum_eiendeler")), F.lit(1.0))
print()
print("%-14s %10s %9s" % ("rel tolerance", "violations", "of filings"))
print("-" * 36)
rel_scan = {}
for t in [1e-9, 1e-6, 1e-4, 1e-3, 1e-2]:
    c = bal.filter(REL > t).count()
    rel_scan[str(t)] = c
    print("%14.0e %10d %8.3f%%" % (t, c, 100.0 * c / n_bal))

# Adopted rule: a filing must fail both tests to count. Absolute alone would
# flag large filings for trivial relative errors; relative alone would flag
# tiny balance sheets for a 1 NOK rounding difference.
TOL_ABS, TOL_REL = 1.5, 1e-6
real = bal.filter((F.abs(DELTA) > TOL_ABS) & (REL > TOL_REL)).cache()
n_real = real.count()

print()
print("Adopted rule: abs > %.1f NOK AND relative > %.0e" % (TOL_ABS, TOL_REL))
print("  violations: %d  (%.3f%% of filings)" % (n_real, 100.0 * n_real / n_bal))
print("  down from %d at the original 0.5 NOK threshold" % n_viol)

findings["tolerance"] = {
    "absolute_scan": tol_scan, "relative_scan": rel_scan,
    "adopted_abs_nok": TOL_ABS, "adopted_rel": TOL_REL,
    "violations_adopted": n_real,
    "violations_original": n_viol,
    "pct_adopted": 100.0 * n_real / n_bal,
}

abs tolerance  violations of filings
------------------------------------
           0.5      15463    3.478%
           1.5       3602    0.810%
           2.5       2682    0.603%
          10.0       2385    0.536%
         100.0       2275    0.512%
        1000.0       1669    0.375%
      100000.0        536    0.121%

rel tolerance  violations of filings
------------------------------------
         1e-09      15267    3.434%
         1e-06       7014    1.577%
         1e-04       2107    0.474%
         1e-03       1761    0.396%
         1e-02       1491    0.335%

Adopted rule: abs > 1.5 NOK AND relative > 1e-06
  violations: 2664  (0.599% of filings)
  down from 15463 at the original 0.5 NOK threshold


## Zero-asset filings

The samples showed filings with `sum_eiendeler` exactly 0 against a
multi-billion equity and liabilities side. That is neither rounding nor a
mapping difference — a balance sheet cannot have no assets and billions in
funding — so it is counted as its own class.

In [9]:
zero_assets = bal.filter((F.col("sum_eiendeler") == 0)
                         & (F.col("sum_egenkapital_gjeld") != 0))
zero_liab = bal.filter((F.col("sum_egenkapital_gjeld") == 0)
                       & (F.col("sum_eiendeler") != 0))
both_zero = bal.filter((F.col("sum_eiendeler") == 0)
                       & (F.col("sum_egenkapital_gjeld") == 0))

n_za, n_zl, n_bz = zero_assets.count(), zero_liab.count(), both_zero.count()
print("assets = 0, equity+liabilities non-zero : %d" % n_za)
print("equity+liabilities = 0, assets non-zero : %d" % n_zl)
print("both zero (consistent, not a violation) : %d" % n_bz)

if n_za:
    print()
    print("Legal forms of the zero-asset filings:")
    for r in (zero_assets.groupBy("organisasjonsform_kode").count()
              .orderBy(F.desc("count")).limit(8).collect()):
        print("   %-8s %6d" % (r["organisasjonsform_kode"], r["count"]))

    q = zero_assets.approxQuantile("sum_egenkapital_gjeld", [0.5, 0.99], 0.0)
    print()
    print("Their equity+liabilities side: median %.0f NOK, p99 %.0f NOK" % (q[0], q[1]))
    print()
    zero_assets.select("organisasjonsnummer", "navn", "organisasjonsform_kode",
                       "sum_egenkapital_gjeld", "sum_egenkapital", "sum_gjeld") \
               .orderBy(F.desc("sum_egenkapital_gjeld")).show(5, truncate=34)

findings["zero_sided"] = {"assets_zero": n_za, "liabilities_zero": n_zl,
                          "both_zero": n_bz}

assets = 0, equity+liabilities non-zero : 369
equity+liabilities = 0, assets non-zero : 99
both zero (consistent, not a violation) : 5629

Legal forms of the zero-asset filings:
   AS          335
   NUF           9
   ENK           6
   FLI           5
   ESEK          5
   DA            3
   SA            2
   STI           2

Their equity+liabilities side: median 30000 NOK, p99 30000000 NOK

+-------------------+-----------------------------+----------------------+---------------------+---------------+---------+
|organisasjonsnummer|                         navn|organisasjonsform_kode|sum_egenkapital_gjeld|sum_egenkapital|sum_gjeld|
+-------------------+-----------------------------+----------------------+---------------------+---------------+---------+
|          926647725|      MERCY OF FIRE HK ART AS|                    AS|                8.1E9|        2.025E9|  6.075E9|
|          835742032|HALLARAUNE JENSSEN HOLDING AS|                    AS|            3.93125E9|       1.2125E

## Accounting regime

`regnskapsregler` was printed by the layout section but never crossed against
the violations, which was an omission: the largest violators in the sample are
ASA filers reporting under IFRS, whose balance sheet does not map onto the
Norwegian template the API populates. If that is the mechanism, the violation
rate should rise sharply with the IFRS regimes.

In [10]:
print("Violation rate by accounting regime, at the adopted tolerance:")
findings["clustering_corrected"] = {
    "regnskapsregler": rate_by("regnskapsregler", tol=TOL_ABS),
}

# smaa_foretak and ASA both looked elevated at the original threshold. Re-run to
# see whether that survives once rounding is excluded, and whether it is really
# a size effect or an IFRS effect wearing size as a disguise.
for col in ["organisasjonsform_kode", "smaa_foretak", "regnskapsaar", "valuta"]:
    findings["clustering_corrected"][col] = rate_by(col, tol=TOL_ABS)

Violation rate by accounting regime, at the adopted tolerance:
regnskapsregler           filings violations      rate
------------------------------------------------------
regnskapslovenAlminneligRegler     442948       3458     0.78%
forenkletAnvendelseIFRS       1476        107     7.25%
IFRS                          222         37    16.67%

organisasjonsform_kode    filings violations      rate
------------------------------------------------------
AS                         403782       3116     0.77%
ESEK                        10324         39     0.38%
BRL                          9995         24     0.24%
STI                          5680         70     1.23%
NUF                          3545         62     1.75%
ENK                          3266         68     2.08%
FLI                          2810         80     2.85%
SA                           1900         24     1.26%
DA                           1211         18     1.49%
ANS                           897          9   

## Decomposition

The original 15,462 split into named causes. Categories are assigned in
precedence order, so each filing is counted once and the parts sum to the
whole.

In [11]:
rounding_only = F.abs(DELTA) <= TOL_ABS
is_zero_sided = ((F.col("sum_eiendeler") == 0) | (F.col("sum_egenkapital_gjeld") == 0))
is_ifrs = F.col("regnskapsregler").isin("IFRS", "forenkletAnvendelseIFRS")

category = (F.when(rounding_only, "rounding_1nok")
            .when(is_zero_sided, "zero_sided")
            .when(is_ifrs, "ifrs_regime")
            .otherwise("unexplained"))

decomp_rows = (viol.withColumn("category", category)
               .groupBy("category")
               .agg(F.count(F.lit(1)).alias("n"),
                    F.expr("percentile_approx(abs(sum_eiendeler - sum_egenkapital_gjeld), 0.5)")
                    .alias("median_delta"))
               .orderBy(F.desc("n")).collect())

print("%-16s %10s %9s %18s" % ("category", "filings", "share", "median delta NOK"))
print("-" * 58)
decomp = {}
for r in decomp_rows:
    decomp[r["category"]] = {"n": r["n"], "pct": 100.0 * r["n"] / n_viol,
                             "median_delta": r["median_delta"]}
    print("%-16s %10d %8.2f%% %18.0f"
          % (r["category"], r["n"], 100.0 * r["n"] / n_viol, r["median_delta"]))

print()
print("total %d  (matches the original count: %s)"
      % (sum(v["n"] for v in decomp.values()),
         sum(v["n"] for v in decomp.values()) == n_viol))

findings["decomposition"] = decomp

category            filings     share   median delta NOK
----------------------------------------------------------
rounding_1nok         11861    76.71%                  1
unexplained            3023    19.55%               1000
zero_sided              435     2.81%              30000
ifrs_regime             144     0.93%               1000

total 15463  (matches the original count: True)


# Part 2: statement layout## Is `oppstillingsplan` really constant?

Checked at all three layers. If it varies in MongoDB but not in our table, we
flattened it wrongly; if it is constant everywhere, the API genuinely returns
one layout for every filer.

In [12]:
print("Analytics table:")
for r in filed.groupBy("oppstillingsplan").count().orderBy(F.desc("count")).collect():
    print("   %-16s %d" % (r["oppstillingsplan"], r["count"]))

print("\nRaw Parquet mirror:")
raw = spark.read.parquet(os.path.join(PARQUET_DIR, "financial_data")).filter(
    "fetch_status = 'success'")
for r in (raw.groupBy(F.col("data")[0]["oppstillingsplan"].alias("plan"))
          .count().orderBy(F.desc("count")).collect()):
    print("   %-16s %d" % (r["plan"], r["count"]))

print("\nMongoDB:")
for r in db.financial_data.aggregate([
        {"$match": {"fetch_status": "success"}},
        {"$unwind": "$data"},
        {"$group": {"_id": "$data.oppstillingsplan", "n": {"$sum": 1}}},
        {"$sort": {"n": -1}}], allowDiskUse=True):
    print("   %-16s %d" % (r["_id"], r["n"]))

# regnskapsregler is the other field that could distinguish accounting regimes.
print("\nregnskapsregler, the other candidate discriminator:")
for r in (filed.groupBy("regnskapsregler").count()
          .orderBy(F.desc("count")).collect()):
    print("   %-16s %d" % (r["regnskapsregler"], r["count"]))

findings["oppstillingsplan"] = {
    "analytics": {r["oppstillingsplan"]: r["count"] for r in
                  filed.groupBy("oppstillingsplan").count().collect()},
    "regnskapsregler": {r["regnskapsregler"]: r["count"] for r in
                        filed.groupBy("regnskapsregler").count().collect()},
}

Analytics table:
   store            444646

Raw Parquet mirror:
   store            445225

MongoDB:
   store            445225

regnskapsregler, the other candidate discriminator:
   regnskapslovenAlminneligRegler 442948
   forenkletAnvendelseIFRS 1476
   IFRS             222


## Industry as a proxy for the income statement

The layout hypothesis was that banks and insurers keep a different income
statement with no `driftsinntekter` line. `oppstillingsplan` cannot test it,
but industry code can: NACE divisions 64 to 66 are financial and insurance
activities, and 64.20 specifically is holding companies.

This also separates the two hypotheses, which the original framing conf lated.

A *bank* has operating revenue and reports it on a different line; a *holding
company* has no operating revenue at all.

Both would show as `revenue_missing`,but only the second is a dormant-entity
story.

In [13]:
div = F.substring(F.col("naeringskode1_kode"), 1, 2)
missing = (F.col("operating_margin_status") == "revenue_missing").cast("int")

by_div = (filed.withColumn("division", div)
          .groupBy("division")
          .agg(F.count(F.lit(1)).alias("n"), F.sum(missing).alias("miss"))
          .filter("n >= 500")
          .withColumn("pct", 100.0 * F.col("miss") / F.col("n")))

base = 100.0 * filed.agg(F.sum(missing)).collect()[0][0] / filed.count()
print("Corpus-wide revenue_missing rate: %.2f%%" % base)

print("\nHighest revenue_missing rate by NACE division (min 500 filings):")
print("%-10s %10s %10s %9s" % ("division", "filings", "missing", "rate"))
print("-" * 42)
top_div = by_div.orderBy(F.desc("pct")).limit(15).collect()
for r in top_div:
    print("%-10s %10d %10d %8.2f%%" % (r["division"], r["n"], r["miss"], r["pct"]))

# Named divisions, so the report can point at them rather than at bare codes.
NAMED = {"64": "Financial service activities (incl. holding companies)",
         "65": "Insurance and pension funding",
         "66": "Activities auxiliary to financial services",
         "68": "Real estate activities",
         "70": "Head office and management consultancy"}
print("\nDivisions of specific interest:")
named_rows = by_div.filter(F.col("division").isin(list(NAMED))).collect()
for r in sorted(named_rows, key=lambda x: -x["pct"]):
    print("  %-4s %-52s %8.2f%%  (n=%d)"
          % (r["division"], NAMED[r["division"]], r["pct"], r["n"]))

findings["revenue_missing_by_division"] = {
    "corpus_rate_pct": base,
    "top15": {r["division"]: {"n": r["n"], "miss": r["miss"], "pct": r["pct"]}
              for r in top_div},
    "named": {r["division"]: {"n": r["n"], "miss": r["miss"], "pct": r["pct"]}
              for r in named_rows},
}

Corpus-wide revenue_missing rate: 20.47%

Highest revenue_missing rate by NACE division (min 500 filings):
division      filings    missing      rate
------------------------------------------
64              15131       8533    56.39%
00              48591      24300    50.01%
50               2221        693    31.20%
94               5373       1484    27.62%
09                610        167    27.38%
72               1335        324    24.27%
63                816        191    23.41%
68              95386      22136    23.21%
60                679        154    22.68%
35               1972        436    22.11%
70              16595       3486    21.01%
82               1932        405    20.96%
62              15673       2774    17.70%
71              15576       2718    17.45%
28                796        136    17.09%

Divisions of specific interest:
  64   Financial service activities (incl. holding companies)    56.39%  (n=15131)
  68   Real estate activities                 

## The positive signature of a dormant or holding company

Absence of revenue is weak evidence on its own. Three things should be true if
these are holding and dormant entities rather than filers using another
layout:1. they employ nobody 2. their result comes from financial rather than
operating income3. they still hold a balance sheet The employee comparison is
the sharpest of the three, because a bank has staffand a shell company does
not.

In [14]:
miss_rows = filed.filter("operating_margin_status = 'revenue_missing'")
comp_rows = filed.filter("operating_margin_status = 'computed'")

def employee_profile(df, label):
    n = df.count()
    zero_or_null = df.filter(F.col("antall_ansatte").isNull()
                             | (F.col("antall_ansatte") == 0)).count()
    med = df.filter(F.col("antall_ansatte").isNotNull()) \
            .approxQuantile("antall_ansatte", [0.5], 0.0)
    print("  %-18s n=%7d   no employees %6.2f%%   median (where recorded) %s"
          % (label, n, 100.0 * zero_or_null / n, med[0] if med else "n/a"))
    return {"n": n, "no_employees_pct": 100.0 * zero_or_null / n,
            "median_employees": med[0] if med else None}

print("Employees:")
emp = {"revenue_missing": employee_profile(miss_rows, "revenue_missing"),
       "computed": employee_profile(comp_rows, "computed")}

# If income is financial rather than operating, netto_finans should be present
# and material far more often among the revenue_missing group.
print("\nFinancial income:")
fin_prof = {}
for df, label in [(miss_rows, "revenue_missing"), (comp_rows, "computed")]:
    n = df.count()
    has_fin = df.filter(F.col("netto_finans").isNotNull()
                        & (F.abs(F.col("netto_finans")) > 0)).count()
    has_bal = df.filter(F.col("sum_eiendeler").isNotNull()
                        & (F.col("sum_eiendeler") > 0)).count()
    fin_prof[label] = {"n": n,
                       "nonzero_netto_finans_pct": 100.0 * has_fin / n,
                       "nonzero_balance_sheet_pct": 100.0 * has_bal / n}
    print("  %-18s non-zero netto_finans %6.2f%%   non-zero balance sheet %6.2f%%"
          % (label, 100.0 * has_fin / n, 100.0 * has_bal / n))

findings["dormant_signature"] = {"employees": emp, "financial": fin_prof}

Employees:
  revenue_missing    n=  91026   no employees  99.12%   median (where recorded) 19.0
  computed           n= 293156   no employees  78.45%   median (where recorded) 12.0

Financial income:
  revenue_missing    non-zero netto_finans  72.96%   non-zero balance sheet  96.70%
  computed           non-zero netto_finans  92.30%   non-zero balance sheet  99.44%


## Persist

In [15]:
path = os.path.join(DATA_DIR, "diagnostics_balance_and_layout.json")
with open(path, "w") as fh:
    json.dump(findings, fh, indent=2, default=str)

print("Wrote", path)
for k in findings:
    print("  ", k)

Wrote /home/jovyan/data/diagnostics_balance_and_layout.json
   tolerance_nok
   checked
   violations
   violation_pct
   mongodb
   magnitude
   rounding
   side_consistency
   clustering
   samples
   tolerance
   zero_sided
   clustering_corrected
   decomposition
   oppstillingsplan
   revenue_missing_by_division
   dormant_signature


## How to read the output

**Provenance first.** If the MongoDB count matches the analytics table, the
discrepancy arrived with the data and everything below characterises it. If it
does not, stop: the fault is in our export or flattening.

**Then the tolerance.** If the deltas are whole numbers with a median of 1, the
original 0.5 NOK threshold was measuring the source's rounding rather than any
error, and only the adopted-tolerance figure belongs in the report. **Then the
decomposition.** `rounding_1nok` is not a data quality problem. `zero_sided` is
a genuine source defect and is worth reporting with an example. `ifrs_regime`
would mean the API maps a different accounting standard onto a Norwegian
template, which is a mapping limitation rather than an error in either the
filing or our pipeline. Whatever remains as `unexplained` is thehonest residual
and should be reported as such rather than attributed. **Layout.** A
`revenue_missing` rate that is elevated in NACE division 64 and inunclassified
entities, while at or below the corpus rate in division 66, argues against the
different-income-statement explanation and for dormant and holding entities.

The employee share is the decisive statistic; financial income is the weaker
test, since a genuinely dormant company has little of either.